<a href="https://colab.research.google.com/github/6pumpkin/SkinModel/blob/main/Corrorsion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#패키지 설치

In [ ]:
# --- 패키지 설치 ---
!pip install rdkit numpy==1.26.4
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0
!pip install torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.3.0+cu121.html
!pip install torch_geometric

#1.라이브러리 임포트

In [ ]:
# ==========================================
# 1. 파이썬 내장 라이브러리 및 통신
# ==========================================
import os
import io
import base64
import pickle
import requests

# ==========================================
# 2. 데이터 처리 및 연산
# ==========================================
import numpy as np

# ==========================================
# 3. PyTorch & PyTorch Geometric (딥러닝 모델)
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import global_add_pool, GATv2Conv, BatchNorm
from torch_scatter import scatter_add

# ==========================================
# 4. RDKit (화학 정보학 및 분자 처리)
# ==========================================
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import MACCSkeys
from rdkit.DataStructs import TanimotoSimilarity
from rdkit.Chem.Draw import MolDraw2DCairo

# ==========================================
# 5. 시각화 및 이미지 처리 (어텐션 맵 렌더링)
# ==========================================
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from PIL import Image

#2. 헬퍼 함수

In [ ]:
class SkinCorrosionPredictor:
    def __init__(self, model_url, artifact_url, device='cpu'):
        # 1. 파일이 로컬에 없으면 깃허브에서 다운로드
        model_local_path = "best_hybrid_model.pth"
        artifact_local_path = "inference_artifacts.pkl"

        if not os.path.exists(model_local_path):
            print("모델 파일 다운로드 중...")
            with open(model_local_path, "wb") as f:
                f.write(requests.get(model_url).content)

        if not os.path.exists(artifact_local_path):
            print("아티팩트 파일 다운로드 중...")
            with open(artifact_local_path, "wb") as f:
                f.write(requests.get(artifact_url).content)

        # 2. 이후 로직은 동일 (다운로드된 파일 로드)
        self.device = torch.device(device)
        with open(artifact_local_path, 'rb') as f:
            self.artifacts = pickle.load(f)

        self.scaler = self.artifacts['scaler']
        # ... (이하 기존과 동일)

ARTIFACT_RAW_URL = "https://github.com/6pumpkin/SkinModel/raw/main/artifact/inference_artifacts_corr.pkl"
MODEL_RAW_URL = "https://github.com/6pumpkin/SkinModel/raw/main/bestmodel/best_hybrid_model_corr.pth"

# 이제 파일 경로 대신 URL을 넘겨줍니다.
predictor = SkinCorrosionPredictor(MODEL_RAW_URL, ARTIFACT_RAW_URL, device='cpu')

In [ ]:
def format_metric(value):
    """
    입력값이 숫자, 텐서, numpy 배열이든 상관없이
    안전하게 소수점 4자리 문자열로 변환하는 함수.
    """
    # 값이 하나뿐인 텐서나 배열일 경우, 숫자(float)만 추출
    if isinstance(value, torch.Tensor) and value.numel() == 1:
        value = value.item()
    elif isinstance(value, np.ndarray) and value.size == 1:
        value = value.item()
    elif isinstance(value, np.ndarray): # 여러 값을 가진 numpy 배열의 경우, 문자열로 변환
        return np.array2string(value, precision=4, separator=', ')

    # 이제 value는 순수한 숫자이므로 안전하게 포맷팅
    if isinstance(value, (int, float)):
        return f"{value:.4f}"
    else:
        # 그 외의 경우는 그냥 문자열로 반환
        return str(value)

###2.1 AttentiveFP 모델 정의

In [ ]:
# RDKit에서 계산 가능한 200여개 디스크립터 목록
RDKIT_DESC_NAMES = [name for name, _ in Descriptors._descList]

def calculate_rdkit_descriptors(smiles_string):
    """SMILES로부터 RDKit 2D 디스크립터 벡터를 계산합니다."""
    mol = Chem.MolFromSmiles(smiles_string)
    if not mol:
        # 계산 불가능 시, 0으로 채워진 벡터 반환
        return np.zeros(len(RDKIT_DESC_NAMES))

    desc = [func(mol) for _, func in Descriptors._descList]
    return np.array(desc, dtype=np.float32)

# =================================================================
# 분자 구조를 위한 피처(Feature) 생성 함수 및 AttentiveFP GAT 모델
# =================================================================

def one_of_k_encoding(x, allowable_set):
    """주어진 집합(allowable_set)에 대해 원-핫 인코딩 벡터를 생성합니다."""
    if x not in allowable_set:
        raise Exception(f"입력 {x}가 허용된 집합 {allowable_set}에 없습니다.")
    return [x == s for s in allowable_set]

def one_of_k_encoding_unk(x, allowable_set):
    """알려지지 않은 입력은 집합의 마지막 요소로 처리하여 원-핫 인코딩합니다."""
    if x not in allowable_set:
        x = allowable_set[-1]
    return [x == s for s in allowable_set]

# 기존 atom_features를 아래 코드로 교체하세요.
def atom_features(atom):
    """
    [수정] 원자 번호와 나머지 특징을 올바르게 결합하는 새로운 특징 생성 함수.
    변수 사용 오류를 수정하고 모든 특징이 정상적으로 포함되도록 했습니다.
    """
    # 1. 원자 번호를 첫 번째 특징으로 사용
    features = [atom.GetAtomicNum()]

    # 2. 나머지 모든 특징을 features 리스트에 순서대로 추가
    features += one_of_k_encoding(atom.GetDegree(), [0, 1, 2, 3, 4, 5])
    features += [atom.GetFormalCharge(), atom.GetNumRadicalElectrons()]
    features += one_of_k_encoding_unk(
        atom.GetHybridization(), [
            Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
            Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
            Chem.rdchem.HybridizationType.SP3D2, 'other'
        ]
    )
    features += [atom.GetIsAromatic()]
    features += one_of_k_encoding_unk(atom.GetTotalNumHs(), [0, 1, 2, 3, 4])

    # 3. 원자가 고리에 속해 있는지 여부 (True/False -> 1/0)
    features.append(atom.IsInRing())

    # 4. 몇 각 고리에 속해 있는지에 대한 원-핫 인코딩 (Multi-hot)
    # (예: 6각 고리에만 속해 있으면 [0,0,0,1,0,0])
    ring_sizes = [3, 4, 5, 6, 7, 8] # 일반적인 고리 크기
    features += [atom.IsInRingSize(s) for s in ring_sizes]

    # 카이랄성 특징 추가
    try:
        features += one_of_k_encoding_unk(
            atom.GetProp('_CIPCode'), ['R', 'S']
        ) + [atom.HasProp('_ChiralityPossible')]
    except:
        features += [False, False] + [atom.HasProp('_ChiralityPossible')]

    return np.array(features)

def bond_features(bond, use_chirality=True):
    """
    RDKit의 결합(Bond) 객체로부터 결합 특징 벡터를 생성합니다.

    포함되는 특징:
    - 결합 종류 (단일, 이중, 삼중, 방향족)
    - 컨쥬게이션 및 고리 여부
    - 입체화학 (Stereochemistry)
    """
    bt = bond.GetBondType()
    bond_feats = [
        bt == Chem.rdchem.BondType.SINGLE, bt == Chem.rdchem.BondType.DOUBLE,
        bt == Chem.rdchem.BondType.TRIPLE, bt == Chem.rdchem.BondType.AROMATIC,
        bond.GetIsConjugated(),
        bond.IsInRing()
    ]

    if use_chirality:
        bond_feats += one_of_k_encoding_unk(
            str(bond.GetStereo()),
            ["STEREONONE", "STEREOANY", "STEREOZ", "STEREOE"]
        )
    return np.array(bond_feats)

def num_atom_features():
    """[수정] atom_features 함수에 맞춰 길이를 계산합니다."""
    m = Chem.MolFromSmiles('c1ccccc1') # 고리 구조를 가진 분자로 테스트
    return len(atom_features(m.GetAtoms()[0]))

def num_bond_features():
    """간단한 분자를 이용해 결합 특징 벡터의 길이를 계산합니다."""
    m = Chem.MolFromSmiles('CC')
    return len(bond_features(m.GetBonds()[0]))

# ================================
# GAT 기반 AttentiveFP 모델 정의
# ================================

class GatedSkipConnection(nn.Module):
    """
    Gated Skip Connection 레이어.
    두 입력(x, Fx)을 받아 학습 가능한 게이트(z)를 통해 조합합니다.
    출력 = z * Fx + (1 - z) * x
    """
    def __init__(self, in_features, out_features):
        super(GatedSkipConnection, self).__init__()
        # 입력과 GAT 출력의 차원이 다를 경우를 대비한 선형 변환 레이어
        if in_features != out_features:
            self.skip_proj = nn.Linear(in_features, out_features)
        else:
            self.skip_proj = nn.Identity()

        # 게이트 z를 계산하기 위한 선형 레이어
        self.gate_linear = nn.Linear(out_features, out_features)

    def forward(self, x, Fx):
        # 1. Skip Connection: 입력 x의 차원을 Fx와 맞춥니다.
        x_proj = self.skip_proj(x)

        # 2. 게이트(z) 계산: x와 Fx를 더한 후, 선형 변환과 시그모이드 함수를 통과시킵니다.
        #    시그모이드를 통해 게이트 값은 항상 0과 1 사이가 됩니다.
        z = torch.sigmoid(self.gate_linear(x_proj + Fx))

        # 3. 게이트를 적용하여 두 출력을 조합합니다.
        return z * Fx + (1 - z) * x_proj

# 기존 GNN 모델의 이름을 GNN_Extractor로 변경하고, forward 함수를 분리합니다.
class GNN_Extractor(nn.Module):
    """
    [리팩토링] GNN 특징 추출기 역할을 하도록 수정된 모델.
    """
    def __init__(self, hidden_channels, out_channels, dropout=0.2):
        super(GNN_Extractor, self).__init__()
        # ... (기존 __init__의 atom_embedding, in_channels 계산 등은 동일) ...
        self.atom_embedding = nn.Embedding(119, 32)
        other_feature_len = num_atom_features() - 1
        in_channels = 32 + other_feature_len
        num_bond_feats = num_bond_features()

        # GAT 레이어 정의 (기존과 동일)
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, dropout=dropout,
                               edge_dim=num_bond_feats, add_self_loops=False)
        self.conv2 = GATv2Conv(hidden_channels * 4, hidden_channels, heads=4, dropout=dropout,
                               edge_dim=num_bond_feats, add_self_loops=False)

        # ✨ --- Gated Skip Connection 레이어 추가 --- ✨
        self.gate1 = GatedSkipConnection(in_channels, hidden_channels * 4)
        self.gate2 = GatedSkipConnection(hidden_channels * 4, hidden_channels * 4)

        self.bn1 = BatchNorm(hidden_channels * 4)
        self.bn2 = BatchNorm(hidden_channels * 4)
        # -------------------------------------------

        # MLP 부분 (기존과 동일)
        self.fc1 = nn.Linear(hidden_channels * 4, hidden_channels)
        self.fc2 = nn.Linear(hidden_channels, out_channels)
        self.dropout_layer = nn.Dropout(p=dropout)

    def extract_graph_features(self, data):
        """ GNN 순전파 로직을 Gated Skip Connection을 사용하도록 수정 """
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        atom_num_features = x[:, 0].long()
        other_features = x[:, 1:]
        atom_embedding_vecs = self.atom_embedding(atom_num_features)
        x_combined = torch.cat([atom_embedding_vecs, other_features], dim=-1)

        # --- ✨ Gated Skip Connection 적용 --- ✨
        # 레이어 1
        x_conv1, (edge_index_1, alpha_1) = self.conv1(x_combined, edge_index, edge_attr=edge_attr, return_attention_weights=True)
        x_norm1 = self.bn1(x_conv1)
        x_gated1 = self.gate1(x_combined, F.elu(x_norm1))
        x_gated1 = self.dropout_layer(x_gated1)

        # 레이어 2
        x_conv2, (edge_index_2, alpha_2) = self.conv2(x_gated1, edge_index_1, edge_attr=edge_attr, return_attention_weights=True)
        x_norm2 = self.bn2(x_conv2)
        x_gated2 = self.gate2(x_gated1, F.elu(x_norm2))
        # ------------------------------------

        x_readout = global_add_pool(x_gated2, batch)
        gnn_feature_vector = F.relu(self.fc1(x_readout))

        return gnn_feature_vector, (edge_index_2, alpha_2)

        return gnn_feature_vector, (edge_index_2, alpha_2)

    def forward(self, data, visualize=False):
        """ GNN 단독으로도 예측을 수행할 수 있는 전체 순전파 함수 """
        gnn_feature_vector, (edge_index_2, alpha_2) = self.extract_graph_features(data)

        # 최종 예측
        x_readout = self.dropout_layer(gnn_feature_vector)
        out = self.fc2(x_readout)

        if visualize:
            attention_weights = (edge_index_2, alpha_2.mean(dim=1))
            return out.squeeze(-1), attention_weights
        else:
            return out.squeeze(-1)


class HybridModel(nn.Module):
    """
    GNN 특징과 머신러닝 디스크립터를 결합하는 하이브리드 모델.
    """
    def __init__(self, gnn_model, num_descriptors, gnn_feature_dim, desc_mlp_dim=64, out_channels=1, dropout=0.2):
        super(HybridModel, self).__init__()

        # 1. GNN 특징 추출기 (미리 초기화된 모델을 받음)
        self.gnn = gnn_model

        # 2. RDKit 디스크립터를 처리할 별도의 MLP
        self.desc_mlp = nn.Sequential(
            nn.Linear(num_descriptors, desc_mlp_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(desc_mlp_dim * 2, desc_mlp_dim)
        )

        # 3. GNN 출력과 MLP 출력을 결합하여 최종 예측을 수행할 Combiner MLP
        self.combiner = nn.Sequential(
            nn.Linear(gnn_feature_dim + desc_mlp_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, out_channels)
        )

    def forward(self, data, visualize=False):
        # 1. GNN 부분: 그래프 특징 추출
        # gnn_feature_vector의 모양: [batch_size, gnn_feature_dim]
        gnn_feature_vector, attention_info = self.gnn.extract_graph_features(data)

        # 2. MLP 부분: 디스크립터 특징 추출
        # DataLoader가 자동으로 desc 속성을 배치 처리해 줌 (data.desc)
        # desc_features의 모양: [batch_size, num_descriptors]
        desc_features = data.desc
        desc_out = self.desc_mlp(desc_features)

        # 3. 특징 결합(Concatenate) 및 최종 예측
        combined_features = torch.cat([gnn_feature_vector, desc_out], dim=1)
        final_out = self.combiner(combined_features)


        # 4. visualize 플래그에 따라 반환값 결정
        if visualize:
            edge_index, alpha = attention_info
            # 4개 헤드의 어텐션 스코어(alpha)를 평균내어 1차원 벡터로 만듭니다.
            attention_scores = alpha.mean(dim=1)

            return final_out.squeeze(-1), (edge_index, attention_scores)
        else:
            return final_out.squeeze(-1)

### 2.2 학습 및 평가, 데이터 처리 함수

In [ ]:
from torch_geometric.data import Data

def smiles_to_graph_hybrid(smiles_string, scaler):
    """
    새로운 SMILES 문자열을 GNN 특징(원자, 결합)과 분자 디스크립터가 결합된
    PyTorch Geometric Data 객체로 변환합니다. (추론용)
    """
    try:
        # 1. 분자 객체 생성
        mol = Chem.MolFromSmiles(smiles_string)
        if mol is None:
            return None

        # 2. GNN을 위한 원자 특징(x) 생성
        atom_feats = [atom_features(atom) for atom in mol.GetAtoms()]
        x = torch.tensor(atom_feats, dtype=torch.float)

        # 3. GNN을 위한 엣지 특징(edge_index, edge_attr) 생성
        if mol.GetNumBonds() > 0:
            bond_feats = [bond_features(bond) for bond in mol.GetBonds()]
            edge_indices = []
            for bond in mol.GetBonds():
                i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
                edge_indices.extend([(i, j), (j, i)])
            edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
            edge_attr = torch.tensor(bond_feats + bond_feats, dtype=torch.float)
        else:
            # 결합이 없는 단일 원자 등의 예외 처리
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr = torch.empty((0, num_bond_features()), dtype=torch.float)

        # 4. 하이브리드 모델을 위한 분자 디스크립터(desc) 계산 및 스케일링 적용
        desc = calculate_rdkit_descriptors(smiles_string)
        desc[~np.isfinite(desc)] = 0  # NaN/inf 예외 처리
        scaled_desc = scaler.transform(desc.reshape(1, -1))

        # 5. 최종 Data 객체 반환 (추론용이므로 정답 y 속성은 할당하지 않음)
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, smiles=smiles_string)
        data.desc = torch.tensor(scaled_desc, dtype=torch.float)

        return data

    except Exception as e:
        # 로그 시스템이 있다면 여기에 기록 (예: logging.error)
        return None

###2.3: 스캐폴드 교차검증 함수

In [ ]:
# (이전에 정의된 HybridModel, train, evaluate 함수가 필요합니다)
# from previous_code import HybridModel, train, evaluate

# ==================================
# 스캐폴드(Scaffold) 기반 데이터 분할 및 교차검증 함수
# ==================================

def generate_scaffold(smiles, include_chirality=False):
    """SMILES 문자열로부터 분자의 Murcko 스캐폴드를 추출합니다."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold, isomericSmiles=include_chirality)
    except:
        return None

def scaffold_split(dataset, n_folds=10):
    """
    화학적 구조(scaffold)를 기반으로 데이터셋을 여러 fold로 분할합니다.
    동일한 scaffold를 가진 분자들은 같은 fold에 속하게 하여, 모델의 일반화 성능을
    더 엄격하게 평가합니다.
    """
    # 1. 분자를 스캐폴드 기준으로 그룹화
    scaffolds = defaultdict(list)
    for i, data in enumerate(dataset):
        scaffold = generate_scaffold(data.smiles)
        # 스캐폴드가 없는 경우, 분자 자체를 키로 사용
        scaffolds[scaffold or data.smiles].append(i)

    # 2. 크기가 큰 스캐폴드 그룹부터 분배하기 위해 정렬
    scaffold_sets = sorted(scaffolds.values(), key=len, reverse=True)

    # 3. 각 fold의 크기가 비슷해지도록 스캐폴드 그룹을 분배
    folds = [[] for _ in range(n_folds)]
    for scaffold_set in scaffold_sets:
        smallest_fold_idx = np.argmin([len(f) for f in folds])
        folds[smallest_fold_idx].extend(scaffold_set)

    return folds

# === [기존 Optuna 탐색 코드 - 주석처리] ===
# print("\n--- 스캐폴드 기반 Optuna 최적화 (5-Fold, 20 Trials) ---")
# best_params, study = run_scaffold_cross_validation(train_dataset, n_folds=5, n_trials=100)

# === [신규] 이전 Optuna 탐색으로 확인한 최적 하이퍼파라미터를 직접 설정 ===
print("\n--- 사전 탐색된 최적 하이퍼파라미터를 직접 사용 ---")

best_params = {
    'hidden'      : 64,    # ← 실제 탐색 결과값으로 교체
    'dropout'     : 0.25935783922584027,    # ← 실제 탐색 결과값으로 교체
    'lr'          : 0.0036566136663614577,  # ← 실제 탐색 결과값으로 교체
    'weight_decay': 0.00021238591265631827,   # ← 실제 탐색 결과값으로 교체
}

print("사용할 하이퍼파라미터:")
for k, v in best_params.items():
    print(f"  {k}: {v}")


###2.4: 적용 범위(AD) 관련 함수

In [ ]:
def smiles_to_maccs_fp(smiles):
    """
    SMILES 문자열을 MACCS 키 분자 지문(fingerprint)으로 변환합니다.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return MACCSkeys.GenMACCSKeys(mol)
    except:
        return None

def check_applicability_domain(new_smiles, train_fps, threshold_S):
    """
    새로운 화합물이 저장된 학습 데이터(train_fps)와 임계값(threshold_S)을 기준으로
    적용 범위(AD) 내에 있는지 판별합니다.
    """
    new_fp = smiles_to_maccs_fp(new_smiles)

    if new_fp is None:
        return "유효하지 않은 SMILES", -1.0
    if not train_fps:
        return "AD 미계산 (기준 데이터 없음)", -1.0

    # 새로운 분자와 학습 데이터셋 분자들 간의 최대 Tanimoto 유사도 계산
    max_sim_to_train = max(TanimotoSimilarity(new_fp, train_fp) for train_fp in train_fps)

    # 저장된 임계값(threshold_S)과 비교하여 최종 판정
    if max_sim_to_train >= threshold_S:
        return "적용 범위 내 (신뢰도 높음)", max_sim_to_train
    else:
        return "적용 범위 밖 (신뢰도 낮음)", max_sim_to_train

###2.5: 시각화 관련 함수 (실제 구현)

In [ ]:
# ── 1. CAS -> SMILES 변환 함수 (PubChem API) ────────────────────
def get_smiles_from_cas(cas_number):
    """
    CAS 번호를 입력받아 PubChem API를 통해 SMILES 문자열을 반환합니다.
    (소프트웨어에서 사용자가 CAS 번호만 입력했을 때를 대비한 편의 기능)
    """
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{cas_number}/property/IsomericSMILES,SMILES/JSON"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            props = resp.json().get('PropertyTable', {}).get('Properties', [{}])[0]
            smi = props.get('IsomericSMILES') or props.get('SMILES')
            return smi
    except Exception as e:
        # 실제 서버 환경에서는 print 대신 로깅(logging.error)을 사용하세요.
        print(f"PubChem API 호출 중 오류 발생: {e}")

    return None

# ── 2. 어텐션 맵 Base64 이미지 생성 함수 ──────────────────────────
def generate_attention_map_base64(smiles_string, model, scaler, device):
    """
    SMILES를 받아 모델의 어텐션 가중치를 계산하고,
    붉은색으로 칠해진 분자 이미지를 Base64 인코딩 문자열로 반환합니다.
    """
    # 1. 그래프 변환
    data_input = smiles_to_graph_hybrid(smiles_string, scaler)
    if data_input is None:
        return None

    # 2. 어텐션 가중치 추출 (모델 추론)
    model.eval()
    with torch.no_grad():
        # visualize=True 옵션을 통해 어텐션 점수를 받아옵니다.
        _, (edge_index, attention_scores) = model(data_input.to(device), visualize=True)

    n_atoms = data_input.x.size(0)

    # 출발 노드 기준으로 가중치 집계 (원자별 어텐션)
    node_weights = scatter_add(src=attention_scores, index=edge_index[0], dim=0, dim_size=n_atoms).squeeze()
    weights_np = node_weights.cpu().numpy()

    # 3. 가중치 정규화 (0.0 ~ 1.0)
    w_min, w_max = weights_np.min(), weights_np.max()
    if abs(w_max - w_min) < 1e-8:
        norm_w = [0.5] * n_atoms # 모든 가중치가 같을 때의 예외 처리
    else:
        norm_w = [(float(w) - w_min) / (w_max - w_min + 1e-8) for w in weights_np]

    # 4. 색상 매핑 (coolwarm 컬러맵 사용)
    cmap_obj = plt.get_cmap('coolwarm')
    norm_obj = Normalize(vmin=0, vmax=1)
    mappable = plt.cm.ScalarMappable(norm=norm_obj, cmap=cmap_obj)

    atom_colors = {}
    for i in range(n_atoms):
        rgba = mappable.to_rgba(float(norm_w[i]))
        atom_colors[i] = tuple(float(x) for x in rgba)

    # 5. 분자 이미지 그리기 (RDKit Cairo 렌더러)
    mol_vis = Chem.MolFromSmiles(smiles_string)
    drawer = MolDraw2DCairo(500, 500)
    drawer.drawOptions().addAtomIndices = True # 원자 번호 표시
    drawer.DrawMolecule(
        mol_vis,
        highlightAtoms=list(range(n_atoms)),
        highlightAtomColors=atom_colors
    )
    drawer.FinishDrawing()

    # 메모리 상에서 PNG 바이트 데이터 추출
    png_bytes = drawer.GetDrawingText()

    # 6. Base64 인코딩
    # 프론트엔드(UI)에서는 <img src="data:image/png;base64,인코딩된문자열"> 형태로 바로 띄울 수 있습니다.
    b64_string = base64.b64encode(png_bytes).decode('utf-8')

    return b64_string

#3.메인 실행 스크립트

In [ ]:
# ── 1. 파일 경로 설정 ──────────────────────────────────────────
# 이전 셀에서 다운로드한 파일의 경로를 사용합니다.
ARTIFACT_PATH = 'inference_artifacts.pkl'
MODEL_PATH = 'best_hybrid_model.pth'

# ── 2. 학습 환경 및 장치 설정 ──────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"추론에 사용할 장치: {device}")

# ── 3. 스케일러 및 아티팩트 파일 로드 ───────────────────────────
print("1. 추론용 아티팩트(스케일러, AD 데이터, 하이퍼파라미터) 로드 중...")
try:
    with open(ARTIFACT_PATH, 'rb') as f:
        artifacts = pickle.load(f)

    # 아티팩트에서 학습 당시의 객체들 추출
    descriptor_scaler = artifacts['scaler']
    train_fingerprints = artifacts['train_fps']
    ad_threshold = artifacts['ad_threshold']
    best_params = artifacts['best_params']
    print("   -> 아티팩트 로드 완료!")
except FileNotFoundError:
    print(f"❌ 아티팩트 파일을 찾을 수 없습니다: {ARTIFACT_PATH}")
    raise

# ── 4. 최적 하이퍼파라미터 기반 모델 구조(Architecture) 재건 ─────
print("\n2. 저장된 최적 하이퍼파라미터로 모델 가상 구조 재건 중...")
gnn_hidden_channels = best_params['hidden']

num_desc = descriptor_scaler.n_features_in_

gnn_extractor = GNN_Extractor(
    hidden_channels=gnn_hidden_channels,
    out_channels=1,
    dropout=best_params['dropout']
).to(device)

final_hybrid_model = HybridModel(
    gnn_model=gnn_extractor,
    num_descriptors=num_desc,
    gnn_feature_dim=gnn_hidden_channels,
    out_channels=1
).to(device)

# ── 5. 최고 성능 가중치(Weights) 주입 ───────────────────────────
print("\n3. 최고 성능 모델 가중치(State Dict) 로드 및 주입 중...")
try:
    final_hybrid_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    final_hybrid_model.eval()  # 🌟 중요: 모델을 평가(추론) 모드로 고정
    print("   -> 가중치 주입 및 모델 준비 완료!")
    print(f"\n✅ [성공] 100회 학습 없이 가중치 로드만으로 즉시 예측할 준비가 끝났습니다.")
except FileNotFoundError:
    print(f"❌ 모델 가중치 파일을 찾을 수 없습니다: {MODEL_PATH}")
    raise

# ── 6. [검증] 로드된 모델로 즉석 예측 함수 정의 및 테스트 ─────────
def predict_skin_corrosion(smiles_string):
    """SMILES를 입력받아 피부 부식성 결과와 AD 판정을 내리는 추론 함수"""
    # 1. 데이터 전처리
    data_input = smiles_to_graph_hybrid(smiles_string, descriptor_scaler)
    if data_input is None:
        return {"error": "유효하지 않은 SMILES 문자열입니다."}

    # 2. AD(적용 범위) 판정 (인자를 3개로 수정!)
    # 기존: check_applicability_domain(smiles_string, train_fingerprints, ad_threshold, smiles_to_maccs_fp)
    # 수정: fp_function 인자를 제거함
    ad_status, sim_score = check_applicability_domain(
        smiles_string, train_fingerprints, ad_threshold
    )

    domain_label = "적용 범위 내 (신뢰도 높음)" if "내" in ad_status else "적용 범위 밖 (신뢰도 낮음)"

    # 3. 모델 예측
    with torch.no_grad():
        logit_val = final_hybrid_model(data_input.to(device))
        prob = torch.sigmoid(logit_val).item()

    pred_label = "Corrosion (양성)" if prob >= 0.5 else "Negative (음성)"

    return {
        "SMILES": smiles_string,
        "예측 결과": pred_label,
        "부식성 확률": f"{prob * 100:.2f}%",
        "비부식성 확률": f"{(1 - prob) * 100:.2f}%",
        "적용 범위(AD) 판정": domain_label,
        "최대 유사도 점수": round(sim_score, 4),
        "AD 임계값": round(ad_threshold, 4)
    }

In [ ]:
# =====================================================================
# [사용자 입력] CAS 번호 또는 SMILES 중 하나만 입력하세요
# =====================================================================
INPUT_CAS    = "69-72-7"   # CAS 번호 입력 (예: 살리실산 69-72-7)
INPUT_SMILES = ""          # SMILES 입력 (CAS 입력 시 "" 로 비움)
# =====================================================================

# --- 1단계: CAS -> SMILES 변환 ---
def cas_to_smiles_pubchem(cas):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{cas}/property/IsomericSMILES,SMILES/JSON"
    resp = requests.get(url, timeout=10)
    if resp.status_code == 200:
        try:
            props = resp.json().get('PropertyTable', {}).get('Properties', [{}])[0]
            smi = props.get('IsomericSMILES') or props.get('SMILES')
            if smi:
                return smi
        except Exception as e:
            print("JSON 데이터 파싱 중 오류 발생:", e)
    return None

# 입력값 처리
if INPUT_SMILES.strip():
    query_smiles = INPUT_SMILES.strip()
elif INPUT_CAS.strip():
    query_smiles = cas_to_smiles_pubchem(INPUT_CAS.strip())
else:
    raise ValueError("INPUT_CAS 또는 INPUT_SMILES 를 입력하세요.")

if query_smiles is None:
    raise ValueError("SMILES 획득 실패. CAS 번호나 SMILES를 확인하세요.")

mol_check = Chem.MolFromSmiles(query_smiles)
if mol_check is None:
    raise ValueError("유효하지 않은 SMILES: " + query_smiles)

print(f"\n🔬 분석 분자 SMILES: {query_smiles}")
print("=" * 60)

# --- 2단계 & 3단계: 독성 예측 및 AD 판정 (앞서 만든 함수 재활용!) ---
# 이전 셀에서 정의한 predict_skin_corrosion 함수를 호출하여 한 번에 끝냅니다.
result_dict = predict_skin_corrosion(query_smiles)

print("\n[1 & 2] 예측 및 AD 판정 결과")
print("-" * 40)
if "error" in result_dict:
    print(result_dict["error"])
else:
    for key, val in result_dict.items():
        if key != "SMILES": # SMILES는 위에서 출력했으므로 생략
            print(f"  {key:<15s} : {val}")


# --- 4단계: Attention Weights 시각화 (Colab 화면 출력용) ---
print("\n[3] Attention Weights 시각화")
print("-" * 40)

# 시각화를 위해 그래프 데이터로 변환 (스케일러는 이전에 로드된 descriptor_scaler 사용)
data_input = smiles_to_graph_hybrid(query_smiles, descriptor_scaler)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with torch.no_grad():
    # visualize=True 옵션으로 어텐션 점수(attn)를 함께 받아옵니다.
    _, (ei, attn) = final_hybrid_model(data_input.to(device), visualize=True)

n_atoms = data_input.x.size(0)
node_w = scatter_add(src=attn, index=ei[0], dim=0, dim_size=n_atoms).squeeze()
weights_np = node_w.cpu().numpy()

mol_vis = Chem.MolFromSmiles(query_smiles)
print("  [원자별 Attention Weight]")
for i, atom in enumerate(mol_vis.GetAtoms()):
    print(f"    Atom {i:2d} ({atom.GetSymbol():2s}): {float(weights_np[i]):.4f}")

# 정규화 (Min-Max Scaling)
w_min, w_max = weights_np.min(), weights_np.max()
if abs(w_max - w_min) < 1e-8:
    norm_w = [0.5] * n_atoms
else:
    norm_w = [(float(w) - w_min) / (w_max - w_min + 1e-8) for w in weights_np]

# 색상 매핑
cmap_obj    = plt.get_cmap('coolwarm')
norm_obj    = Normalize(vmin=0, vmax=1)
mappable    = plt.cm.ScalarMappable(norm=norm_obj, cmap=cmap_obj)
atom_colors = {i: tuple(float(x) for x in mappable.to_rgba(float(norm_w[i]))) for i in range(n_atoms)}

# 분자 그리기 (RDKit)
drawer = MolDraw2DCairo(500, 500)
drawer.drawOptions().addAtomIndices = True
drawer.DrawMolecule(
    mol_vis,
    highlightAtoms=list(range(n_atoms)),
    highlightAtomColors=atom_colors
)
drawer.FinishDrawing()
png_bytes = drawer.GetDrawingText()

# 화면 출력 (Matplotlib)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(Image.open(io.BytesIO(png_bytes)))
ax.set_axis_off()

cbar = fig.colorbar(mappable, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)
cbar.set_label('Normalized Attention Weight', rotation=270, labelpad=15)

plt.tight_layout()
plt.show()
print("\n✅ 분석 완료")